# HW4 Part 2: MLflow Experiment Tracking

Logs 2 runs to the `olist-satisfaction` experiment, registers the best model to Production.

In [1]:
import time
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from mlflow import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

print('Imports OK')

Imports OK


In [ ]:
# Load data 
df = pd.read_csv('ml_ready_df.csv')

TARGET = 'is_positive_review'
X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_cols = [
    'delivery_days', 'delivery_vs_estimated', 'price', 'freight_value',
    'seller_score', 'num_previous_sales', 'cust_reviews', 'freight_ratio',
    'num_previous_reviews', 'num_items',
    'same_state', 'is_repeat_customer', 'delivery_missing',
]
cat_cols = [
    'product_category_name_english', 'seller_state', 'payment_type'
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

Train: 93,949  |  Test: 23,488


In [ ]:
# Preprocessor 
def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
    ])

print('Preprocessor factory ready')

Preprocessor factory ready


In [ ]:
# MLflow setup 
# Store experiments locally in mlruns/
mlflow.set_tracking_uri('mlruns')
experiment_name = 'olist-satisfaction'
mlflow.set_experiment(experiment_name)

print(f'Tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment   : {experiment_name}')

/Users/margarida/Documents/Fairfield University/01. MSBA/02. Spring 26/DATA 6545/Homework/HW4/hw4-mlops/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/19 16:29:41 INFO mlflow.tracking.fluent: Experiment with name 'olist-satisfaction' does not exist. Creating a new experiment.


Tracking URI : mlruns
Experiment   : olist-satisfaction


In [ ]:
# Two runs to log 
# Run 1: default HistGradientBoosting  (baseline)
# Run 2: tuned  HistGradientBoosting  (best params from HW2 GridSearchCV)

runs_config = [
    {
        'run_name': 'hist-gb-default',
        'params': {
            'model_type':    'HistGradientBoostingClassifier',
            'learning_rate': 0.1,
            'max_depth':     None,
            'max_iter':      100,
            'class_weight':  'balanced',
        },
        'classifier': HistGradientBoostingClassifier(
            class_weight='balanced', random_state=42
        ),
    },
    {
        'run_name': 'hist-gb-tuned',
        'params': {
            'model_type':    'HistGradientBoostingClassifier',
            'learning_rate': 0.05,
            'max_depth':     5,
            'max_iter':      300,
            'class_weight':  'balanced',
        },
        'classifier': HistGradientBoostingClassifier(
            learning_rate=0.05, max_depth=5, max_iter=300,
            class_weight='balanced', random_state=42
        ),
    },
]

run_ids = {}

for config in runs_config:
    with mlflow.start_run(run_name=config['run_name']) as run:

        # Log parameters
        for k, v in config['params'].items():
            mlflow.log_param(k, v)
        mlflow.log_param('train_size',  len(X_train))
        mlflow.log_param('n_features',  len(numeric_cols) + len(cat_cols))

        # Build and fit pipeline
        pipe = Pipeline([
            ('preprocessor', make_preprocessor()),
            ('classifier',   config['classifier']),
        ])
        start = time.time()
        pipe.fit(X_train, y_train)
        training_time = time.time() - start

        # Evaluate
        y_pred  = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]

        metrics = {
            'accuracy':  round(accuracy_score(y_test, y_pred),             4),
            'precision': round(precision_score(y_test, y_pred),            4),
            'recall':    round(recall_score(y_test, y_pred),               4),
            'f1':        round(f1_score(y_test, y_pred),                   4),
            'roc_auc':   round(roc_auc_score(y_test, y_proba),             4),
            'training_time_seconds': round(training_time, 2),
        }

        # Log metrics
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # Log model artifact
        mlflow.sklearn.log_model(pipe, 'model')

        run_ids[config['run_name']] = run.info.run_id

        print(f"\n{'='*55}")
        print(f"Run : {config['run_name']}")
        print(f"  Accuracy  : {metrics['accuracy']}")
        print(f"  Precision : {metrics['precision']}")
        print(f"  Recall    : {metrics['recall']}")
        print(f"  F1        : {metrics['f1']}")
        print(f"  ROC-AUC   : {metrics['roc_auc']}")
        print(f"  Time      : {metrics['training_time_seconds']}s")
        print(f"  Run ID    : {run.info.run_id}")

print(f"\nAll runs logged to experiment: '{experiment_name}'")

2026/04/19 16:29:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 16:29:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Run : hist-gb-default
  Accuracy  : 0.8157
  Precision : 0.8717
  Recall    : 0.8848
  F1        : 0.8782
  ROC-AUC   : 0.8279
  Time      : 2.06s
  Run ID    : ed597a0fe7ad467ca0d2739daa0bffbd


2026/04/19 16:29:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 16:29:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Run : hist-gb-tuned
  Accuracy  : 0.8165
  Precision : 0.8722
  Recall    : 0.8854
  F1        : 0.8787
  ROC-AUC   : 0.827
  Time      : 3.44s
  Run ID    : 32049b9a9f0e4f4bb4ebb507179bf28e

All runs logged to experiment: 'olist-satisfaction'


In [ ]:
# Register best model (tuned run) to Production 
client     = MlflowClient()
model_name = 'olist-satisfaction-predictor'
best_run_id = run_ids['hist-gb-tuned']

# Register
model_uri        = f'runs:/{best_run_id}/model'
registered_model = mlflow.register_model(model_uri, model_name)
version          = registered_model.version
print(f'Registered: {model_name}  version {version}')

# Transition to Production
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage='Production'
)
print(f'Version {version} → Production')

# Confirm
versions = client.search_model_versions(f"name='{model_name}'")
for v in versions:
    print(f'  Version {v.version}: stage={v.current_stage}')

/Users/margarida/Documents/Fairfield University/01. MSBA/02. Spring 26/DATA 6545/Homework/HW4/hw4-mlops/venv/lib/python3.10/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'olist-satisfaction-predictor'.
2026/04/19 16:29:56 WARNING mlflow.tracking._model_registry.fluent: Run with id 32049b9a9f0e4f4bb4ebb507179bf28e has no artifacts at artifact path 'model', registering model based on models:/m-f45c3ced006242dd815daaf8338d633b instead


Registered: olist-satisfaction-predictor  version 1
Version 1 → Production
  Version 1: stage=Production


Created version '1' of model 'olist-satisfaction-predictor'.
/var/folders/wv/mc9k5j6n3ndc114hpfq2y2fh0000gs/T/ipykernel_97149/731521751.py:13: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
